# Biometria: pipeline no Colab

Antes: `biometria_colab.zip` em `MyDrive/biometria_colab/` (gerado por `scripts/make_colab_bundle.py`).
Use uma GPU com bfloat16 (A100, L4 ou H100). Rode as células em ordem. Se a sessão cair, rode tudo de novo: o que já terminou é restaurado do Drive e pulado.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.cuda.get_device_name(0), "| bfloat16:", torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), "Troque para A100, L4 ou H100 (Ambiente de execução > Alterar tipo)."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['DRIVE'] = DRIVE = '/content/drive/MyDrive/biometria_colab'
assert os.path.exists(f'{DRIVE}/biometria_colab.zip'), 'Coloque biometria_colab.zip em MyDrive/biometria_colab/'

In [ ]:
%%bash
set -e
mkdir -p /content/biometria && cd /content/biometria
[ -f config.yaml ] || unzip -q "$DRIVE/biometria_colab.zip"
command -v rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync)
if [ -d "$DRIVE/sync/results" ]; then rsync -a "$DRIVE/sync/" ./ && echo "progresso anterior restaurado"; fi
ls

In [ ]:
%%bash
set -e
cd /content/biometria
pip -q install uv
uv sync -q --extra cu124 --extra dev
uv run --extra cu124 python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"
uv run --extra cu124 pytest -q 2>&1 | tail -2

In [ ]:
%%bash
cat > /content/sync.sh <<'EOF'
cd /content/biometria
while true; do
  mkdir -p "$DRIVE/sync"
  rsync -a --prune-empty-dirs --include='*/' --include='results/***' --include='artigo/***'     --include='runs/convnext_tiny_gabor/*/best.pt' --exclude='*.pt' --include='runs/***' --exclude='*'     ./ "$DRIVE/sync/"
  sleep 120
done
EOF
pgrep -f /content/sync.sh >/dev/null || (nohup bash /content/sync.sh > /content/sync.log 2>&1 &)
echo "sync ativo"

In [ ]:
%%bash
cd /content/biometria
date -u "+[%H:%M:%S] start on $(nvidia-smi --query-gpu=name --format=csv,noheader)" | tee -a "$DRIVE/pipeline.log"
STAGE=gpu SKIP_PREP=1 PYTHONUNBUFFERED=1 bash scripts/run_pipeline.sh 2>&1 | tee -a "$DRIVE/pipeline.log"   | grep --line-buffered -E '^\[|^\{"epoch": (1|20|40|60|80),|Traceback|Error'   | sed -u -E 's/, "val_eer_threshold".*//'
echo "exit code: ${PIPESTATUS[0]}" | tee -a "$DRIVE/pipeline.log"

In [ ]:
%%bash
cd /content/biometria
rm -f "$DRIVE/colab_results.zip"
zip -qr "$DRIVE/colab_results.zip" results artigo/figures runs -x '*.pt'
ls -la "$DRIVE"